# llama.cpp coding-model benchmark on Kaggle (GPU T4 x2)

One-off benchmark, not a server: it installs a pinned llama.cpp, downloads two public GGUF models in turn, measures
generation speed, prompt-processing speed (2k to 30k tokens), prefix-cache reuse, tool calling (OpenAI format),
the Anthropic `/v1/messages` endpoint, reasoning cost and VRAM on both cards, all over localhost, then writes
`results.json` and `summary.txt` to `/kaggle/working` and exits. No tunnel and no secrets.

`MODE` in the first code cell is a placeholder replaced when the kernel is pushed: `probe` (CPU, environment only),
`smoke` (CPU, tiny model, shrunken sizes, same code) or `gpu` (the real run). An internal 100 minute budget makes it
wrap up with partial results.

In [ ]:
# Cell 1: mode, budget, environment facts. Cheap checks only; nothing is installed or downloaded here.
import atexit, json, os, platform, random, re, shutil, statistics, subprocess, sys, threading, time, traceback, uuid
import urllib.error, urllib.request
from pathlib import Path

T0 = time.time()
MODE = "@MODE@"        # "probe" | "smoke" | "gpu"; the push step replaces the placeholder in a staged copy
assert MODE in ("probe", "smoke", "gpu"), f"MODE placeholder was not replaced: {MODE!r}"
GPU = MODE == "gpu"
SMOKE = MODE != "gpu"
RUN_BENCH = MODE in ("smoke", "gpu")

LLAMA_TAG = "b12430"                      # pinned: the docs and flags below were read at this tag (2026-11-16)
BUDGET_S = {"probe": 15 * 60, "smoke": 12 * 60, "gpu": 92 * 60}[MODE]     # soft: steps are skipped when it is near
HARD_S = int(os.environ.get("LLMBENCH_HARD_S") or {"probe": 15 * 60, "smoke": 15 * 60, "gpu": 100 * 60}[MODE])
CTX = {"probe": 0, "smoke": 4096, "gpu": 32768}[MODE]
CTX_BIG = {"probe": 0, "smoke": 8192, "gpu": 65536}[MODE]
PP_SIZES = [200, 500, 1000, 2000] if SMOKE else [2000, 8000, 16000, 30000]
CACHE_TOKENS = 1500 if SMOKE else 16000
BIG_TOKENS = 2000 if SMOKE else 62000
LOAD_TIMEOUT = 3 * 60 if SMOKE else 25 * 60
# smoke is tiny: one trial per measurement, about 32 output tokens
GEN_TOKENS = 32 if SMOKE else 400
GEN_TOPICS = 1 if SMOKE else 3
TOOL_N = 1 if SMOKE else 5
TOOL_MAX = 96 if SMOKE else 3000
LOOP_TURNS = 4 if SMOKE else 8
PORT = 8080
KEY = "bench-" + uuid.uuid4().hex[:16]     # throwaway key for the local server only

if GPU:
    MODELS = {
        "A": {"repo": "unsloth/Qwen3.6-35B-A3B-GGUF", "file": "Qwen3.6-35B-A3B-UD-Q4_K_M.gguf",
              "size": 22134528992, "big_ctx": True},
        "B": {"repo": "unsloth/Qwen3.8-27B-GGUF", "file": "Qwen3.8-27B-UD-Q4_K_M.gguf",
              "size": 16564259744, "big_ctx": False},
    }
else:    # smoke: tiny public models, same code path
    MODELS = {
        "A": {"repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF", "file": "qwen2.5-0.5b-instruct-q4_k_m.gguf",
              "size": 491400032, "big_ctx": True},
        "B": {"repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF", "file": "qwen2.5-0.5b-instruct-q8_0.gguf",
              "size": 675710816, "big_ctx": False},
    }

OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("llmbench_out")
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = {"mode": MODE, "llama_tag": LLAMA_TAG, "env": {}, "install": {}, "models": {},
           "config": {"ctx": CTX, "ctx_big": CTX_BIG, "pp_sizes": PP_SIZES, "cache_tokens": CACHE_TOKENS,
                      "budget_s": BUDGET_S, "models": MODELS}}


CHILDREN = []          # every subprocess we start (servers, downloads): the watchdog and the exit hook kill them


def _watchdog():
    """Hard limit: whatever hangs (a request, a download, a load), at HARD_S the results are saved, every child is
    killed and the process exits. The soft budget (left()) only skips steps between steps."""
    while time.time() - T0 < HARD_S:
        time.sleep(2)
    print(f"\nWATCHDOG: hard limit of {HARD_S}s reached; saving, killing children, exiting", flush=True)
    try:
        RESULTS["watchdog"] = f"hard limit {HARD_S}s reached at {time.time() - T0:.0f}s"
        save()
        txt = json.dumps({k: RESULTS.get(k) for k in ("install", "models")}, default=str)
        print(txt[:6000], flush=True)
        (OUT / "summary.txt").write_text("WATCHDOG ended the run.\n" + txt[:20000], encoding="utf-8")
    except Exception as e:
        print("watchdog save failed:", e, flush=True)
    for p in CHILDREN:
        try:
            p.kill()
        except Exception:
            pass
    sys.stdout.flush()
    os._exit(3)


def log(*a):
    print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)


def left() -> float:
    return BUDGET_S - (time.time() - T0)


def save():
    (OUT / "results.json").write_text(json.dumps(RESULTS, indent=1, default=str), encoding="utf-8")


threading.Thread(target=_watchdog, daemon=True).start()
log("watchdog armed: soft budget", BUDGET_S, "s, hard limit", HARD_S, "s")


def fatal(msg: str):
    RESULTS["fatal"] = msg
    save()
    print("FATAL:", msg, flush=True)
    raise RuntimeError(msg)


def sh(cmd, timeout=120, env=None):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout, env=env)
    return p.returncode, ((p.stdout or "") + (p.stderr or "")).strip()


# ---- environment facts
E = RESULTS["env"]
E["python"] = sys.version.split()[0]
E["platform"] = platform.platform()
for key, cmd in (("os_release", "grep PRETTY_NAME /etc/os-release"), ("glibc", "ldd --version | head -1"),
                 ("cpus", "nproc"), ("mem", "free -g | head -2"), ("gcc", "gcc --version | head -1"),
                 ("cmake", "cmake --version | head -1"), ("nvcc", "nvcc --version | tail -1"),
                 ("df", "df -h /tmp /kaggle/working /kaggle/temp 2>&1 | head -8")):
    try:
        E[key] = sh(cmd)[1]
    except Exception as e:
        E[key] = f"error: {e}"
for k, v in E.items():
    print(f"{k}: {v}")

# ---- GPU inventory (nvidia-smi)
if GPU:
    rc, full = sh(["nvidia-smi"])
    print(full)
    E["nvidia_smi"] = full
    rc, q = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version,compute_cap",
                "--format=csv,noheader,nounits"])
    E["gpus"] = [x.strip() for x in q.splitlines() if x.strip()] if rc == 0 else []
    if len(E["gpus"]) != 2 or not all("T4" in g for g in E["gpus"]):
        fatal(f"expected 2 x T4, nvidia-smi says: {E['gpus']!r} (is the kernel pushed with machine_shape NvidiaTeslaT4?)")
    E["driver"] = E["gpus"][0].split(",")[3].strip()
else:
    E["gpus"] = []
    E["driver"] = None

# ---- scratch dir: not /kaggle/working (output cap ~20 GB, and everything in it is kept as output)
need = (max(m["size"] for m in MODELS.values()) if RUN_BENCH else 0) + 4_000_000_000
WORK = None
for cand in (os.environ.get("LLMBENCH_WORK"), "/kaggle/temp", "/tmp/llmbench"):
    if not cand:
        continue
    p = Path(cand)
    try:
        p.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage(p).free
    except OSError:
        continue
    print(f"scratch candidate {p}: {free / 1e9:.1f} GB free")
    if free >= need:
        WORK = p
        break
if WORK is None:
    fatal(f"no scratch directory with {need / 1e9:.0f} GB free")
E["scratch"] = str(WORK)
E["scratch_free_gb"] = round(shutil.disk_usage(WORK).free / 1e9, 1)
log("mode", MODE, "scratch", WORK, "budget min", BUDGET_S / 60)
save()


In [ ]:
# Cell 2: get a llama-server binary. GPU: the official Ubuntu CUDA 12.8 release (+ CUDA runtime libs if the system lacks them),
# falling back to a source build for sm_75 only if the prebuilt one cannot run. CPU smoke/probe: the official CPU release,
# and the CUDA tarball is only inspected (glibc, missing libs, --version) so the GPU run holds no surprises.
I = RESULTS["install"]
BASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
CPU_TGZ = f"llama-{LLAMA_TAG}-bin-ubuntu-x64.tar.gz"
CUDA_TGZ = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_TGZ = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
@atexit.register
def _kill_children():
    for p in CHILDREN:
        try:
            if p.poll() is None:
                p.kill()
        except Exception:
            pass


def download(name: str) -> Path:
    dest = WORK / name
    t = time.time()
    rc, out = sh(["curl", "-fL", "--retry", "3", "--retry-delay", "3", "--connect-timeout", "20", "-sS",
                  "-o", str(dest), f"{BASE}/{name}"], timeout=900)
    if rc:
        raise RuntimeError(f"download {name} failed: {out[-300:]}")
    I.setdefault("download_s", {})[name] = round(time.time() - t, 1)
    log(f"downloaded {name} {dest.stat().st_size / 1e6:.0f} MB in {time.time() - t:.0f}s")
    return dest


def untar(tgz: Path, dest: Path):
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    rc, out = sh(["tar", "-xzf", str(tgz), "-C", str(dest)], timeout=300)
    if rc:
        raise RuntimeError(f"extract {tgz.name}: {out[-300:]}")
    tgz.unlink()


def extract(tgz: Path, dest: Path) -> Path:
    untar(tgz, dest)
    exe = [p for p in dest.rglob("llama-server") if p.is_file()]
    if not exe:
        raise RuntimeError(f"no llama-server in {tgz.name}")
    return exe[0]


def lib_env(bindir: Path) -> dict:
    return dict(os.environ, LD_LIBRARY_PATH=f"{bindir}:{os.environ.get('LD_LIBRARY_PATH', '')}")


def inspect_build(server: Path) -> dict:
    """Missing shared libraries (libcuda.so is the driver's, so it is expected to be absent on a CPU kernel),
    glibc/libstdc++ symbol errors, and whether `llama-server --version` runs at all."""
    bindir = server.parent
    env = lib_env(bindir)
    miss, symbols = {}, []
    for f in [server, *bindir.glob("libggml-cuda*.so*")]:
        try:
            p = subprocess.run(["ldd", str(f)], capture_output=True, text=True, env=env)
        except OSError:                      # no ldd (local dev on Windows)
            continue
        for line in (p.stdout + p.stderr).splitlines():
            if "not found" in line and "=>" in line:
                lib = line.split("=>")[0].strip()
                if not lib.startswith("libcuda.so"):
                    miss.setdefault(f.name, []).append(lib)
            elif "version `" in line:
                symbols.append(line.strip()[:200])
    rc, out = sh([str(server), "--version"], timeout=60, env=env)
    return {"missing_libs": miss, "symbol_errors": symbols[:5], "version_rc": rc, "version_out": out[-600:]}


def add_cudart(server: Path):
    d = WORK / "cudart"
    untar(download(CUDART_TGZ), d)
    n = 0
    for f in d.rglob("*.so*"):
        if f.is_file():
            shutil.copy2(f, server.parent / f.name)
            n += 1
    I["cudart_copied_files"] = n
    shutil.rmtree(WORK / "cudart", ignore_errors=True)


def prepare_cuda_prebuilt() -> tuple:
    server = extract(download(CUDA_TGZ), WORK / "cuda")
    info = inspect_build(server)
    I["cuda_prebuilt"] = info
    if info["missing_libs"]:
        log("missing libs in CUDA build, adding the cudart bundle:", info["missing_libs"])
        add_cudart(server)
        info = inspect_build(server)
        I["cuda_prebuilt_after_cudart"] = info
    ok = not info["missing_libs"] and not info["symbol_errors"] and info["version_rc"] == 0
    return server, ok


def build_from_source(configure_only: bool = False) -> Path:
    """Fallback: sm_75-only CUDA build with the toolkit on this machine. The sm_75 target is real code, so it needs no PTX JIT."""
    if not shutil.which("nvcc"):
        raise RuntimeError("no nvcc on this machine: cannot build llama.cpp from source")
    src, build = WORK / "llama.cpp", WORK / "llama.cpp" / "build"
    if not src.exists():
        rc, out = sh(["git", "clone", "-q", "--depth", "1", "--branch", LLAMA_TAG,
                      "https://github.com/ggml-org/llama.cpp", str(src)], timeout=600)
        if rc:
            raise RuntimeError(f"git clone failed: {out[-300:]}")
    t = time.time()
    rc, out = sh(["cmake", "-S", str(src), "-B", str(build), "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                  "-DGGML_NATIVE=OFF", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"], timeout=900)
    I["source_configure_s"] = round(time.time() - t, 1)
    I["source_configure_tail"] = out[-400:]
    if rc:
        raise RuntimeError(f"cmake configure failed: {out[-600:]}")
    if configure_only:
        return None
    t = time.time()
    rc, out = sh(["cmake", "--build", str(build), "--config", "Release", "--target", "llama-server",
                  "-j", str(os.cpu_count() or 4)], timeout=45 * 60)
    I["source_build_s"] = round(time.time() - t, 1)
    if rc:
        raise RuntimeError(f"cmake build failed: {out[-800:]}")
    return build / "bin" / "llama-server"


SERVER = None
t_install = time.time()
if os.environ.get("LLMBENCH_SERVER"):                 # local dev hook: use an existing binary
    SERVER = Path(os.environ["LLMBENCH_SERVER"])
    I["method"] = "LLMBENCH_SERVER"
elif GPU:
    major = int(str(E["driver"]).split(".")[0])
    I["driver_major"] = major
    prebuilt_ok = False
    if major >= 570:
        SERVER, prebuilt_ok = prepare_cuda_prebuilt()
        I["method"] = "prebuilt cuda-12.8" if prebuilt_ok else "prebuilt unusable"
    else:
        I["method"] = f"driver {major} < 570: cannot JIT the PTX in the CUDA 12.8 release"
    if not prebuilt_ok:
        log("falling back to a source build:", I["method"])
        try:
            SERVER = build_from_source()
            I["method"] = "source build sm_75"
        except Exception as e:
            fatal(f"no usable llama-server for the GPU run ({I['method']}): {e}")
else:
    # CPU: the binary the smoke test runs, and a look at the CUDA release for the GPU run
    cpu_server = extract(download(CPU_TGZ), WORK / "cpu")
    I["cpu_build"] = inspect_build(cpu_server)
    SERVER = cpu_server
    I["method"] = "prebuilt cpu"
    try:
        _, ok = prepare_cuda_prebuilt()
        I["cuda_prebuilt_ok_on_this_host"] = ok
    except Exception as e:
        I["cuda_prebuilt_error"] = str(e)[:400]
    try:
        build_from_source(configure_only=True)
        I["source_configure_ok"] = True
    except Exception as e:
        I["source_configure_ok"] = False
        I["source_configure_error"] = str(e)[:400]

I["install_s"] = round(time.time() - t_install, 1)
if RUN_BENCH:
    if SERVER is None or not Path(SERVER).exists():
        fatal("no llama-server binary")
    info = inspect_build(Path(SERVER))
    I["server_inspect"] = info
    if info["version_rc"] != 0 or info["missing_libs"]:
        fatal(f"llama-server does not run here: {info}")
    I["server_path"] = str(SERVER)
    log("llama-server ready:", I["method"], "in", I["install_s"], "s;", info["version_out"].splitlines()[-1:] or "")
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_xet]"], timeout=300)
    I["pip_hf_rc"] = rc
print(json.dumps(I, indent=1, default=str)[:4000])
save()


In [ ]:
# Cell 3: helpers: HTTP to the local server, prompt builder, VRAM sampler, model download, server start/stop.
BASEURL = f"http://127.0.0.1:{PORT}"


def http(method, path, body=None, headers=None, timeout=900, auth=True):
    h = {"Content-Type": "application/json"}
    if auth:
        h["Authorization"] = f"Bearer {KEY}"
    h.update(headers or {})
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(BASEURL + path, data=data, method=method, headers=h)
    t0 = time.time()
    timeout = min(timeout, max(30, left() + 30))          # never wait past the budget
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            raw = r.read()
            return r.status, (json.loads(raw) if raw else None), time.time() - t0
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", "replace")[:2000], time.time() - t0


def timings_of(j: dict) -> dict:
    t = j.get("timings") or {}
    keys = ("prompt_n", "cache_n", "prompt_ms", "prompt_per_second", "predicted_n", "predicted_ms",
            "predicted_per_second")
    return {k: (round(t[k], 2) if isinstance(t.get(k), float) else t.get(k)) for k in keys}


def chat(messages, max_tokens, thinking=None, tools=None, extra=None, timeout=1800):
    """One non-streaming /v1/chat/completions call; returns the server's own timings, usage and the message."""
    body = {"model": "bench", "messages": messages, "max_tokens": max_tokens, "cache_prompt": True}
    if thinking is False:
        body["chat_template_kwargs"] = {"enable_thinking": False}
    if tools:
        body["tools"] = tools
    if extra:
        body.update(extra)
    st, j, wall = http("POST", "/v1/chat/completions", body, timeout=timeout)
    if st != 200:
        raise RuntimeError(f"HTTP {st}: {str(j)[:400]}")
    ch = j["choices"][0]
    return {"wall_s": round(wall, 2), "timings": timings_of(j), "usage": j.get("usage"),
            "finish": ch.get("finish_reason"), "message": ch.get("message") or {}}


def stream_chat(body: dict, timeout=1800) -> dict:
    """Streaming call; times the first reasoning / content / tool-call delta (seconds since the request was sent)."""
    body = dict(body, stream=True, cache_prompt=True)
    req = urllib.request.Request(BASEURL + "/v1/chat/completions", data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"})
    ev = {"first_any_s": None, "first_reasoning_s": None, "first_content_s": None, "first_tool_s": None,
          "reasoning_chunks": 0, "content_chunks": 0, "tool_name": None, "tool_args": "", "finish": None,
          "timings": None}
    t0 = time.time()
    timeout = min(timeout, max(30, left() + 30))
    with urllib.request.urlopen(req, timeout=timeout) as r:
        for raw in r:
            if time.time() - t0 > timeout:                # overall deadline (the socket timeout is per read)
                ev["timed_out"] = True
                break
            line = raw.decode("utf-8", "replace").strip()
            if not line.startswith("data:"):
                continue
            payload = line[5:].strip()
            if payload == "[DONE]":
                break
            j = json.loads(payload)
            if j.get("timings"):
                ev["timings"] = timings_of(j)
            for ch in j.get("choices") or []:
                d, now = ch.get("delta") or {}, round(time.time() - t0, 2)
                if ev["first_any_s"] is None and (d.get("content") or d.get("reasoning_content") or d.get("tool_calls")):
                    ev["first_any_s"] = now
                if d.get("reasoning_content"):
                    ev["first_reasoning_s"] = ev["first_reasoning_s"] if ev["first_reasoning_s"] is not None else now
                    ev["reasoning_chunks"] += 1
                if d.get("content"):
                    ev["first_content_s"] = ev["first_content_s"] if ev["first_content_s"] is not None else now
                    ev["content_chunks"] += 1
                for tc in d.get("tool_calls") or []:
                    ev["first_tool_s"] = ev["first_tool_s"] if ev["first_tool_s"] is not None else now
                    fn = tc.get("function") or {}
                    ev["tool_name"] = fn.get("name") or ev["tool_name"]
                    ev["tool_args"] += fn.get("arguments") or ""
                if ch.get("finish_reason"):
                    ev["finish"] = ch["finish_reason"]
    ev["total_s"] = round(time.time() - t0, 2)
    return ev


# ---- prompts of a known size, different every time (random words behind a unique id: nothing can be cache-reused)
VOCAB = ("time year people way day man thing woman life child world school state family student group country "
         "problem hand part place case week company system program question work government number night point home "
         "water room mother area money story fact month lot right study book eye job word business issue side kind "
         "head house service friend father power hour game line end member law car city community name president "
         "team minute idea kid body information back parent face others level office door health person art war "
         "history party result change morning reason research girl guy moment air teacher force education").split()


def tok_count(text: str) -> int:
    st, j, _ = http("POST", "/tokenize", {"content": text}, timeout=120)
    if st != 200:
        raise RuntimeError(f"/tokenize HTTP {st}: {str(j)[:200]}")
    return len(j["tokens"])


def make_text(target: int) -> tuple:
    """Text of about `target` tokens (within 1 percent), per the server's own tokenizer."""
    rng = random.Random(uuid.uuid4().int)
    ident = f"Document {uuid.uuid4().hex}. "
    nw = max(10, int(target / 1.2))
    text, n = "", 0
    for _ in range(6):
        words = [rng.choice(VOCAB) if rng.random() < 0.92 else str(rng.randint(0, 99999)) for _ in range(nw)]
        text = ident + " ".join(words)
        n = tok_count(text)
        if abs(n - target) <= max(6, target * 0.01):
            break
        nw = max(10, int(nw * target / n))
    return text, n


# ---- VRAM
def vram_used() -> list:
    if not GPU:
        return []
    rc, out = sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=30)
    return [int(x) for x in out.split() if x.strip().isdigit()] if rc == 0 else []


class VramSampler(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.peak, self.ev = [], threading.Event()

    def run(self):
        while not self.ev.is_set():
            v = vram_used()
            if v:
                self.peak = v if not self.peak else [max(a, b) for a, b in zip(self.peak, v)]
            self.ev.wait(2.0)

    def finish(self) -> list:
        self.ev.set()
        self.join(10)
        return self.peak


def wait_vram_free(limit_mib=800, timeout=90):
    end = time.time() + timeout
    while GPU and time.time() < end:
        v = vram_used()
        if v and max(v) < limit_mib:
            return True
        time.sleep(3)
    return not GPU


# ---- model download (to the scratch dir, never /kaggle/working)
DL = ("import sys; from huggingface_hub import hf_hub_download; "
      "p = hf_hub_download(sys.argv[1], sys.argv[2], local_dir=sys.argv[3]); print('PATH', p)")


def tree_bytes(d: Path) -> int:
    return sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) if d.exists() else 0


def download_once(spec: dict, dest: Path, env: dict, deadline: float):
    """One download attempt in a child process; returns (ok, message). Kills it on a stall or at the deadline."""
    p = subprocess.Popen([sys.executable, "-c", DL, spec["repo"], spec["file"], str(dest)], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    CHILDREN.append(p)
    last_t = time.time()
    last_n = tree_bytes(dest)
    last_growth = time.time()
    while p.poll() is None:
        time.sleep(15)
        n = tree_bytes(dest)
        now = time.time()
        if n > last_n:
            last_growth = now
        log(f"download {spec['file']}: {n / 1e9:.2f} of {spec['size'] / 1e9:.2f} GB, "
            f"{(n - last_n) / max(now - last_t, 1) / 1e6:.0f} MB/s")
        last_t, last_n = now, n
        if now - last_growth > 240 or now > deadline:
            p.kill()
            return False, f"download stalled or too slow ({n / 1e9:.2f} GB)"
    out = p.stdout.read()
    path = dest / spec["file"]
    if p.returncode != 0 or not path.exists():
        return False, f"rc {p.returncode}: {out[-300:]}"
    if path.stat().st_size != spec["size"]:
        return False, f"{path.name}: {path.stat().st_size} bytes, expected {spec['size']}"
    return True, str(path)


def download_model(spec: dict, dest: Path) -> Path:
    """Up to 3 attempts. The first run died with HTTP 429 from the Hugging Face CDN (xet) on the second model, so:
    attempt 1 uses the default xet transfer (no high-performance mode), attempts 2 and 3 plain HTTP after a pause."""
    dest.mkdir(parents=True, exist_ok=True)
    deadline = time.time() + (25 * 60 if GPU else 3 * 60)
    errors = []
    for attempt, (pause, plain) in enumerate([(0, False), (60, True), (120, True)], 1):
        if attempt > 1:
            log(f"download attempt {attempt - 1} failed ({errors[-1][:160]}); waiting {pause}s")
            time.sleep(pause)
        if time.time() > deadline:
            break
        env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_DISABLE_TELEMETRY="1")
        if plain:
            env["HF_HUB_DISABLE_XET"] = "1"
        for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):      # public models: never send a token
            env.pop(k, None)
        ok, msg = download_once(spec, dest, env, deadline)
        if ok:
            return Path(msg)
        errors.append(msg)
    raise RuntimeError("download failed after retries: " + " | ".join(e[-250:] for e in errors))


# ---- the server
CURRENT = {"srv": None}


class Server:
    def __init__(self, model: Path, ctx: int, tag: str):
        self.model, self.ctx, self.tag = model, ctx, tag
        self.logfile = WORK / f"server_{tag}.log"
        self.proc = None
        self.args = [str(SERVER), "-m", str(model), "-c", str(ctx), "-ngl", "all" if GPU else "0", "-sm", "layer",
                     "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0", "-np", "1", "--jinja", "--api-key", KEY,
                     "--host", "127.0.0.1", "--port", str(PORT), "--no-webui", "-fit", "off", "--alias", "bench"]

    def log_text(self) -> str:
        try:
            return self.logfile.read_text(errors="replace")
        except OSError:
            return ""

    def start(self) -> float:
        env = lib_env(Path(SERVER).parent)
        env["CUDA_CACHE_PATH"] = str(WORK / "nvcache")          # PTX JIT results (sm_75 is PTX-only in the release)
        env["CUDA_CACHE_MAXSIZE"] = "4294967296"
        self.f = open(self.logfile, "wb")
        self.proc = subprocess.Popen(self.args, stdout=self.f, stderr=subprocess.STDOUT, env=env)
        CHILDREN.append(self.proc)
        CURRENT["srv"] = self
        t0 = time.time()
        while True:
            if self.proc.poll() is not None:
                raise RuntimeError(f"llama-server exited with {self.proc.returncode} while loading; log tail:\n"
                                   + "\n".join(self.log_text().splitlines()[-25:]))
            try:
                st, _, _ = http("GET", "/health", timeout=5)
                if st == 200:
                    return time.time() - t0
            except Exception:
                pass
            if time.time() - t0 > LOAD_TIMEOUT:
                self.stop()
                raise RuntimeError(f"llama-server not ready after {LOAD_TIMEOUT}s; log tail:\n"
                                   + "\n".join(self.log_text().splitlines()[-25:]))
            time.sleep(3)

    def alive(self) -> bool:
        return self.proc is not None and self.proc.poll() is None

    def highlights(self) -> list:
        pat = re.compile(r"offload|KV buffer|compute buffer|model buffer|RS buffer|recurrent|n_ctx|split|"
                         r"CUDA\d|Device \d|flash|memory breakdown|ggml_cuda_init|fit", re.I)
        return [l.strip()[:220] for l in self.log_text().splitlines() if pat.search(l)][:60]

    def stop(self):
        if self.proc is not None and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(60)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(30)
        try:
            self.f.close()
        except Exception:
            pass
        dst = OUT / "logs"
        dst.mkdir(exist_ok=True)
        lines = self.log_text().splitlines()
        keep = lines if len(lines) <= 1500 else lines[:500] + ["... trimmed ..."] + lines[-1000:]
        (dst / self.logfile.name).write_text("\n".join(keep), encoding="utf-8")


In [ ]:
# Cell 4: the measurements. Each returns a dict; every number comes from the server's own `timings` or from wall clocks.
TOOLS = [{"type": "function", "function": {
    "name": "run_command",
    "description": "Run a shell command on the user's Linux machine and return its output and exit code.",
    "parameters": {"type": "object", "properties": {"cmd": {"type": "string", "description": "The shell command line."}},
                   "required": ["cmd"]}}}]
SYSTEM = ("You are a command-line co-pilot on the user's Linux machine. Use the run_command tool to inspect or change "
          "things. Run one command at a time and read its output before choosing the next step. When the task is "
          "done, answer in plain text and mention anything that failed.")
TOOL_TASKS = ["Show me the five largest files under /var/log.",
              "How much free disk space is there on the root filesystem?",
              "Which process is listening on port 8080?",
              "Count the lines of all .py files in the current directory.",
              "Create a directory called backup in my home folder."]
TOPICS = ["the history of the bicycle", "how a refrigerator works", "the causes of the French Revolution"]


def user_msgs(text):
    return [{"role": "user", "content": text}]


def step_warmup():
    text, n = make_text(300 if SMOKE else 1500)
    r = chat(user_msgs(text + "\n\nReply with the single word OK."), 16, thinking=False)
    return {"prompt_tokens": n, "wall_s": r["wall_s"]}


def step_generation():
    runs = []
    for topic in TOPICS[:GEN_TOPICS]:
        r = chat(user_msgs(f"Write a very long, detailed essay (at least 800 words) about {topic}."), GEN_TOKENS,
                 thinking=False, timeout=300)
        t = r["timings"]
        runs.append({"topic": topic, "predicted_n": t["predicted_n"], "tokens_per_s": t["predicted_per_second"],
                     "prompt_n": t["prompt_n"], "wall_s": r["wall_s"], "finish": r["finish"]})
    rates = [x["tokens_per_s"] for x in runs if x["tokens_per_s"]]
    return {"runs": runs, "mean_tokens_per_s": round(statistics.mean(rates), 2) if rates else None}


def pp_call(tokens, max_tokens=1):
    text, n = make_text(tokens)
    r = chat(user_msgs(text + "\n\nReply with the single word OK."), max_tokens, thinking=False, timeout=2400)
    t = r["timings"]
    return {"target": tokens, "prompt_tokens_text": n, "prompt_n": t["prompt_n"], "cache_n": t["cache_n"],
            "prompt_ms": t["prompt_ms"], "tokens_per_s": t["prompt_per_second"], "wall_s": r["wall_s"]}, text


def step_pp(tokens):
    return lambda: pp_call(tokens)[0]


def valid_call(msg: dict):
    """(ok, cmd, why): a tool call to run_command whose arguments parse as JSON with a non-empty string `cmd`."""
    tcs = msg.get("tool_calls") or []
    if not tcs:
        return False, None, "no tool_calls"
    fn = tcs[0].get("function") or {}
    if fn.get("name") != "run_command":
        return False, None, f"wrong name {fn.get('name')!r}"
    try:
        args = json.loads(fn.get("arguments") or "")
    except ValueError:
        return False, None, "arguments are not JSON"
    if not isinstance(args, dict) or not isinstance(args.get("cmd"), str) or not args["cmd"].strip():
        return False, None, "no string cmd"
    return True, args["cmd"], "ok"


def step_tools():
    trials = []
    for task in TOOL_TASKS[:TOOL_N]:
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
        try:
            r = chat(msgs, TOOL_MAX, tools=TOOLS, timeout=600)
        except Exception as e:
            trials.append({"task": task, "ok": False, "why": f"request failed: {e}"[:300]})
            continue
        ok, cmd, why = valid_call(r["message"])
        trials.append({"task": task, "ok": ok, "why": why, "cmd": cmd, "finish": r["finish"], "wall_s": r["wall_s"],
                       "completion_tokens": (r["usage"] or {}).get("completion_tokens"),
                       "reasoning_chars": len(r["message"].get("reasoning_content") or ""),
                       "content_head": (r["message"].get("content") or "")[:120]})
    return {"ok_count": sum(1 for t in trials if t["ok"]), "of": len(trials), "trials": trials}


LISTING = ("total 28\n-rw-r--r-- 1 ops ops  412 Nov 14 09:12 README.md\ndrwxr-xr-x 2 ops ops 4096 Nov 14 09:12 src\n"
           "drwxr-xr-x 2 ops ops 4096 Nov 14 09:12 tests\n-rw-r--r-- 1 ops ops   96 Nov 14 09:12 requirements.txt\n[exit code 0]")
PYTEST_FAIL = ("============ test session starts ============\ncollected 13 items\n\ntests/test_api.py ......\n"
               "tests/test_db.py ..F....\n\nFAILED tests/test_db.py::test_connect - ConnectionRefusedError: [Errno 111] "
               "Connection refused\n======== 1 failed, 12 passed in 3.41s ========\n[exit code 1]")
TEST_DB_SRC = ("import psycopg2\n\ndef test_connect():\n    c = psycopg2.connect(host='localhost', port=5432, dbname='app')\n"
               "    assert c.status\n[exit code 0]")
PG_DOWN = ("* postgresql.service - PostgreSQL database server\n   Active: inactive (dead) since Fri 2026-11-13 22:01:07 "
           "UTC; 11h ago\n[exit code 3]")


def fake_run(cmd: str, state: dict) -> str:
    """Scripted machine, keyed on the command: a bare `pytest` is not installed (the one failure); `python -m pytest`
    runs and one test fails on a refused database connection; the postgres service is down."""
    c = cmd.strip()
    if re.search(r"(^|[;&|(]\s*)pytest\b", c) and "python" not in c:
        return "bash: line 1: pytest: command not found\n[exit code 127]"
    if "pytest" in c or "unittest" in c:
        return PYTEST_FAIL
    if re.search(r"systemctl|service |pg_isready|postgres|psql|5432|netstat|ss -", c):
        return PG_DOWN
    if re.search(r"test_db|conftest|tests/", c) and re.search(r"cat|head|grep|sed|less", c):
        return TEST_DB_SRC
    if re.match(r"(ls|find|tree)\b", c) or (c.startswith("cd ") and "ls" in c):
        return LISTING
    return "[exit code 0]"


def step_loop():
    """Simulated agent loop: the tool outputs are scripted (listing, command not found, test failure, service down)."""
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "The test suite in /srv/app should pass. Run it and tell me whether it does, "
                                        "and if it does not, why."}]
    calls, final, transcript, wall = 0, None, [], 0.0
    for turn in range(LOOP_TURNS):
        r = chat(msgs, TOOL_MAX, tools=TOOLS, timeout=600)
        wall += r["wall_s"]
        m = r["message"]
        tcs = m.get("tool_calls") or []
        if not tcs:
            final = m.get("content") or ""
            transcript.append({"turn": turn, "final": final[:1200], "finish": r["finish"]})
            break
        msgs.append({"role": "assistant", "content": m.get("content") or "", "tool_calls": tcs})
        for tc in tcs:
            ok, cmd, why = valid_call({"tool_calls": [tc]})
            out = fake_run(cmd or "", {})
            transcript.append({"turn": turn, "cmd": cmd, "valid": ok, "fed_back": out.splitlines()[0][:80]})
            msgs.append({"role": "tool", "tool_call_id": tc.get("id") or f"call_{calls}", "content": out})
            calls += 1
    mentions = bool(final and re.search(r"(?i)not found|127|fail|connection|refused|postgres|not running|inactive", final))
    kept_going = calls >= 3
    return {"tool_calls": calls, "kept_going_after_failure": kept_going, "final_answer": bool(final),
            "final_mentions_failure": mentions, "pass": bool(kept_going and final and mentions),
            "total_wall_s": round(wall, 1), "transcript": transcript}


def step_reasoning():
    task = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": TOOL_TASKS[0]}]
    base = {"model": "bench", "messages": task, "tools": TOOLS, "max_tokens": TOOL_MAX}
    default = stream_chat(base, timeout=600)
    nothink = stream_chat(dict(base, chat_template_kwargs={"enable_thinking": False}), timeout=600)
    return {"default": default, "thinking_off": nothink,
            "emits_reasoning_by_default": default["reasoning_chunks"] > 0,
            "default_first_tool_s": default["first_tool_s"], "off_first_tool_s": nothink["first_tool_s"]}


def step_anthropic():
    body = {"model": "bench", "max_tokens": 1500, "system": SYSTEM,
            "messages": [{"role": "user", "content": "List the files in /tmp by calling the run_command tool."}],
            "tools": [{"name": "run_command", "description": TOOLS[0]["function"]["description"],
                       "input_schema": TOOLS[0]["function"]["parameters"]}]}
    hdrs = {"x-api-key": KEY, "anthropic-version": "2023-06-01"}
    st, j, wall = http("POST", "/v1/messages", body, headers=hdrs, timeout=900, auth=False)
    out = {"http": st, "wall_s": round(wall, 2), "auth": "x-api-key"}
    if st == 401:     # the key may be expected as a bearer token instead
        st, j, wall = http("POST", "/v1/messages", body, headers=hdrs, timeout=900, auth=True)
        out.update({"http": st, "wall_s": round(wall, 2), "auth": "x-api-key + bearer"})
    if st == 200 and isinstance(j, dict):
        blocks = j.get("content") or []
        out.update({"type": j.get("type"), "stop_reason": j.get("stop_reason"),
                    "block_types": [b.get("type") for b in blocks],
                    "has_tool_use": any(b.get("type") == "tool_use" for b in blocks),
                    "usage": j.get("usage")})
        out["pass"] = j.get("type") == "message" and bool(blocks)
    else:
        out.update({"pass": False, "body": str(j)[:300]})
    return out


def step_cache():
    """A prompt, the same prompt plus a few hundred tokens, and the exact same prompt again."""
    base, n = make_text(CACHE_TOKENS)
    extra, ne = make_text(200)
    tail = "\n\nReply with the single word OK."
    first = chat(user_msgs(base + tail), 8, thinking=False, timeout=2400)
    second = chat(user_msgs(base + "\n" + extra + tail), 8, thinking=False, timeout=2400)
    third = chat(user_msgs(base + "\n" + extra + tail), 8, thinking=False, timeout=2400)

    def pick(r):
        t = r["timings"]
        return {"prompt_n": t["prompt_n"], "cache_n": t["cache_n"], "prompt_ms": t["prompt_ms"], "wall_s": r["wall_s"]}
    return {"base_tokens": n, "added_tokens": ne, "first_cold": pick(first), "second_plus_200": pick(second),
            "third_identical": pick(third),
            "reevaluated_on_second": second["timings"]["prompt_n"], "reused_on_second": second["timings"]["cache_n"],
            "second_latency_s": second["wall_s"], "first_latency_s": first["wall_s"]}


def run_step(R, name, fn, need_s):
    need_s = need_s * (0.1 if SMOKE else 1)               # the smoke steps are tiny
    if left() < need_s:
        R["steps"][name] = {"skipped": f"time budget: {left():.0f}s left, step needs about {need_s}s"}
        log("SKIP", name, R["steps"][name]["skipped"])
        save()
        return
    srv = CURRENT["srv"]
    if srv is not None and not srv.alive():
        R["steps"][name] = {"error": "llama-server is no longer running",
                            "log_tail": "\n".join(srv.log_text().splitlines()[-15:])}
        save()
        return
    t = time.time()
    log("step", name)
    try:
        out = fn()
    except Exception as e:
        out = {"error": f"{type(e).__name__}: {e}"[:600], "trace": traceback.format_exc()[-1200:]}
    out["step_wall_s"] = round(time.time() - t, 1)
    R["steps"][name] = out
    save()


In [ ]:
# Cell 5: per model: download, serve at 32k, measure, (model A) restart at 64k, delete the model.
def run_model(key: str, spec: dict):
    R = RESULTS["models"][key] = {"repo": spec["repo"], "file": spec["file"], "steps": {}}
    mdir = WORK / "models" / key
    srv = None
    try:
        t = time.time()
        log(f"model {key}: downloading {spec['repo']}/{spec['file']}")
        path = download_model(spec, mdir)
        R["download_s"] = round(time.time() - t, 1)
        R["download_gb"] = round(path.stat().st_size / 1e9, 2)
        save()

        srv = Server(path, CTX, f"{key}_{CTX}")
        R["server_args"] = [a if a != KEY else "<key>" for a in srv.args]
        try:
            R["load_s"] = round(srv.start(), 1)
        except Exception as e:
            R["load_error"] = str(e)[:1500]
            save()
            return
        time.sleep(3)
        R["vram_after_load_mib"] = vram_used()
        R["log_highlights"] = srv.highlights()
        st, props, _ = http("GET", "/props", timeout=30)
        if st == 200 and isinstance(props, dict):
            R["props"] = {"n_ctx": (props.get("default_generation_settings") or {}).get("n_ctx"),
                          "build_info": props.get("build_info"), "total_slots": props.get("total_slots")}
        log(f"model {key}: loaded in {R['load_s']}s at ctx {CTX}, VRAM MiB per card {R['vram_after_load_mib']}")
        save()

        sampler = VramSampler()
        sampler.start()
        run_step(R, "warmup", step_warmup, 60)
        run_step(R, "generation", step_generation, 300)
        run_step(R, "tools", step_tools, 900)
        run_step(R, "loop", step_loop, 600)
        run_step(R, "reasoning", step_reasoning, 400)
        run_step(R, "anthropic", step_anthropic, 150)
        for n in PP_SIZES:
            run_step(R, f"pp_{n}", step_pp(n), 240 if n <= 16000 else 600)
        run_step(R, "cache", step_cache, 900)
        R["vram_peak_during_tests_mib"] = sampler.finish()
        save()
        srv.stop()
        wait_vram_free()

        if spec["big_ctx"] and CTX_BIG:
            R["ctx_big"] = {"ctx": CTX_BIG}
            if left() < (15 * 60 if GPU else 90):
                R["ctx_big"]["skipped"] = f"time budget: {left():.0f}s left"
            else:
                srv = Server(path, CTX_BIG, f"{key}_{CTX_BIG}")
                try:
                    R["ctx_big"]["load_s"] = round(srv.start(), 1)
                    time.sleep(3)
                    R["ctx_big"]["vram_after_load_mib"] = vram_used()
                    R["ctx_big"]["log_highlights"] = srv.highlights()
                    sampler = VramSampler()
                    sampler.start()
                    tmp = {"steps": {}}
                    run_step(tmp, "big_prompt", lambda: pp_call(BIG_TOKENS, 16)[0], 900)
                    R["ctx_big"]["big_prompt"] = tmp["steps"]["big_prompt"]
                    R["ctx_big"]["vram_peak_mib"] = sampler.finish()
                except Exception as e:
                    R["ctx_big"]["error"] = str(e)[:1500]
                save()
                srv.stop()
                wait_vram_free()
    finally:
        if srv is not None:
            srv.stop()
        shutil.rmtree(mdir, ignore_errors=True)           # free the disk before the next model
        save()


def run_all():
    order = ("B", "A") if GPU else ("A", "B")        # run 2 of the GPU benchmark: B first (its download failed in run 1)
    for i, key in enumerate(order):
        spec = MODELS[key]
        if i == 1 and left() < (25 * 60 if GPU else 120):
            RESULTS["models"][key] = {"skipped": f"time budget: {left():.0f}s left"}
            save()
            continue
        try:
            run_model(key, spec)
        except Exception as e:
            RESULTS["models"].setdefault(key, {})["error"] = f"{type(e).__name__}: {e}"[:1500]
            RESULTS["models"][key]["trace"] = traceback.format_exc()[-1500:]
            save()
    RESULTS["total_wall_s"] = round(time.time() - T0, 1)
    save()


if RUN_BENCH:
    run_all()
else:
    RESULTS["total_wall_s"] = round(time.time() - T0, 1)
    save()


In [ ]:
# Cell 6: summary (last cell output), results.json, make sure nothing is left running.
for p in CHILDREN:
    try:
        if p.poll() is None:
            p.terminate()
    except Exception:
        pass


def g(d, *path, default=None):
    for k in path:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def fmt_tps(x):
    return f"{x:.1f}" if isinstance(x, (int, float)) else "n/a"


lines = [f"llm bench, mode {MODE}, llama.cpp {LLAMA_TAG}, total {time.time() - T0:.0f}s",
         f"install: {g(RESULTS, 'install', 'method')} in {g(RESULTS, 'install', 'install_s')}s; driver {E.get('driver')}; "
         f"gpus {E.get('gpus')}"]
if RESULTS.get("fatal"):
    lines.append("FATAL: " + str(RESULTS["fatal"])[:500])
for key, R in RESULTS["models"].items():
    lines.append("")
    spec = MODELS.get(key, {})
    lines.append(f"== model {key}: {spec.get('repo')} / {spec.get('file')}")
    if R.get("skipped") or R.get("error") or R.get("load_error"):
        lines.append("   " + str(R.get("skipped") or R.get("error") or R.get("load_error"))[:400])
        if "load_s" not in R:
            continue
    S = R.get("steps", {})
    lines.append(f"   download {R.get('download_s')}s ({R.get('download_gb')} GB), load {R.get('load_s')}s at ctx {CTX}, "
                 f"VRAM after load {R.get('vram_after_load_mib')} MiB, peak in tests {R.get('vram_peak_during_tests_mib')}")
    lines.append(f"   generation: {fmt_tps(g(S, 'generation', 'mean_tokens_per_s'))} tok/s "
                 f"{[x.get('tokens_per_s') for x in g(S, 'generation', 'runs', default=[])]}")
    pp = []
    for n in PP_SIZES:
        s = S.get(f"pp_{n}", {})
        pp.append(f"{n}: {fmt_tps(s.get('tokens_per_s'))} tok/s ({s.get('wall_s', s.get('skipped') or s.get('error'))}s)")
    lines.append("   prompt processing: " + "; ".join(pp))
    c = S.get("cache", {})
    lines.append(f"   cache: 2nd call re-evaluated {c.get('reevaluated_on_second')} tokens, reused "
                 f"{c.get('reused_on_second')}, latency {c.get('second_latency_s')}s vs cold {c.get('first_latency_s')}s; "
                 f"identical repeat re-evaluated {g(c, 'third_identical', 'prompt_n')}"
                 + (f" [{c.get('error') or c.get('skipped')}]" if c.get("error") or c.get("skipped") else ""))
    t = S.get("tools", {})
    lines.append(f"   tool calls: {t.get('ok_count')}/{t.get('of')} valid"
                 + (f" [{t.get('error') or t.get('skipped')}]" if t.get("error") or t.get("skipped") else ""))
    lp = S.get("loop", {})
    lines.append(f"   loop: pass={lp.get('pass')} calls={lp.get('tool_calls')} kept_going={lp.get('kept_going_after_failure')} "
                 f"final_mentions_failure={lp.get('final_mentions_failure')}"
                 + (f" [{lp.get('error') or lp.get('skipped')}]" if lp.get("error") or lp.get("skipped") else ""))
    a = S.get("anthropic", {})
    lines.append(f"   anthropic /v1/messages: pass={a.get('pass')} http={a.get('http')} blocks={a.get('block_types')}")
    r = S.get("reasoning", {})
    lines.append(f"   reasoning by default: {r.get('emits_reasoning_by_default')}, first tool call after "
                 f"{r.get('default_first_tool_s')}s (thinking off: {r.get('off_first_tool_s')}s), reasoning chunks "
                 f"{g(r, 'default', 'reasoning_chunks')}")
    b = R.get("ctx_big")
    if b:
        lines.append(f"   ctx {b.get('ctx')}: load {b.get('load_s')}s, VRAM after load {b.get('vram_after_load_mib')} MiB, "
                     f"peak {b.get('vram_peak_mib')}, big prompt {fmt_tps(g(b, 'big_prompt', 'tokens_per_s'))} tok/s "
                     f"({g(b, 'big_prompt', 'prompt_n')} tokens)"
                     + (f" [{b.get('error') or b.get('skipped')}]" if b.get("error") or b.get("skipped") else ""))
    errs = [f"{k}: {v.get('error') or v.get('skipped')}" for k, v in S.items() if v.get("error") or v.get("skipped")]
    if errs:
        lines.append("   step problems: " + " | ".join(e[:160] for e in errs))
SUMMARY = "\n".join(lines)
RESULTS["summary"] = SUMMARY
save()
(OUT / "summary.txt").write_text(SUMMARY, encoding="utf-8")
print(SUMMARY)
